In [ ]:
import os
import sys
import torch
import pickle
import pandas as pd
from pathlib import Path
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
import torch.nn.functional as F
import glob
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from Net.GNN_Grid_Search.GAT import load_training_data

# Add the project root to the Python path
project_root = Path(__file__).resolve().parent.parent
sys.path.append(str(project_root))

# Define the model class
class GATConv(torch.nn.Module):
    def __init__(self, input_dim, hidden_channels, dropout, heads):
        super(GATConv, self).__init__()
        self.conv1 = GATv2Conv(input_dim, hidden_channels, heads=heads, concat=True, add_self_loops=False)
        self.dropout = torch.nn.Dropout(dropout)
        self.pooling_function = global_mean_pool
        self.out_layer = torch.nn.Linear(hidden_channels * heads, 1)

    def forward(self, data):
        edge_index = data.edge_index
        batch = data.batch
        x = data.x

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.pooling_function(x, batch)
        x = self.out_layer(x)
        return x

# Load the dataset
def load_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir)
    pkl_path = dataset_dir / "data_list_0.pkl"
    
    if not pkl_path.exists():
        raise FileNotFoundError(f"Dataset file not found: {pkl_path}")

    with open(pkl_path, "rb") as f:
        data_list = pickle.load(f)

    print(f"Loaded dataset: {pkl_path} ({len(data_list)} graphs)")
    return data_list

# Predict and save results
def predict_and_save(model, data_list, output_csv):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    predictions = []

    loader = DataLoader(data_list, batch_size=32, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch).view(-1)

            for i in range(len(batch.y)):
                predictions.append({
                    "PDB_ID": batch.PDB_ID[i],
                    "Residue_Number": batch.Residue_Number[i],
                    "Residue": batch.Residue_Name[i],
                    "True_pKa": batch.y[i].item(),
                    "Predicted_pKa": out[i].item()
                })

    df = pd.DataFrame(predictions)
    df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")

    return predictions

def inspect_checkpoint(model_path):
    """Inspect the structure of the checkpoint file."""
    checkpoint = torch.load(model_path, map_location=torch.device("cpu"))
    print(f"Keys in the checkpoint: {list(checkpoint.keys())}")
    return checkpoint

# Predict and save results for all models
def predict_with_all_models(model_dir, data_sets, output_dir):
    device = torch.device("cpu")
    os.makedirs(output_dir, exist_ok=True)

    # Find all .pth files in the directory
    model_paths = sorted(glob.glob(os.path.join(model_dir, "*.pth")))

    metrics = []  # To store MAE and RMSE for each model and dataset

    for dataset_idx, data_list in enumerate(data_sets):
        print(f"Processing Dataset {dataset_idx + 1}...")

        for model_path in model_paths:
            print(f"Loading model: {model_path}")

            # Inspect the checkpoint structure
            checkpoint = inspect_checkpoint(model_path)

            # Load model
            model = GATConv(input_dim=data_list[0].x.shape[1], hidden_channels=48, dropout=0.3, heads=6)

            if "model_state_dict" in checkpoint:
                model.load_state_dict(checkpoint["model_state_dict"])
            else:
                print("Warning: 'model_state_dict' not found. Attempting to load the entire checkpoint.")
                model.load_state_dict(checkpoint)

            model.to(device)
            model.eval()

            # Predict and save
            model_name = os.path.basename(model_path).replace(".pth", "")
            output_csv = os.path.join(output_dir, f"predictions_dataset_{dataset_idx + 1}_{model_name}.csv")
            predictions = predict_and_save(model, data_list, output_csv)

            # Calculate MAE and RMSE
            true_labels = [pred["True_pKa"] for pred in predictions]
            predicted_labels = [pred["Predicted_pKa"] for pred in predictions]
            mae = mean_absolute_error(true_labels, predicted_labels)
            rmse = np.sqrt(mean_squared_error(true_labels, predicted_labels))

            metrics.append({"Dataset": dataset_idx + 1, "Model": model_name, "MAE": mae, "RMSE": rmse})
            print(f"Dataset {dataset_idx + 1}, Model: {model_name}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")

    # Save metrics to a CSV file
    metrics_df = pd.DataFrame(metrics)
    metrics_csv = os.path.join(output_dir, "model_metrics_all_datasets.csv")
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"Metrics saved to {metrics_csv}")

if __name__ == "__main__":
    # Paths
    model_dir = "/home/ziyu-song/Graph_pKa/Results/GAT_Grid_Search/loss_MSELoss_h48_b16_lr0.01_d0.3_hd6"
    dataset_dir = "/home/ziyu-song/Graph_pKa/Data_1/4_Residues_W_Local_Frame/Subsets"
    output_dir = "/home/ziyu-song/Graph_pKa/Results/All_Predictions"

    # Load all datasets
    data_sets, input_dim, Residue_Type_labels = load_training_data(dataset_dir)

    # Predict with all models and datasets
    predict_with_all_models(model_dir, data_sets, output_dir)

In [ ]:
"""Utility to extract predicted pKa values from PROPKA `.pka` outputs.

The script walks a directory containing PROPKA output files, harvests the
"SUMMARY OF THIS PREDICTION" table from each file, and writes a consolidated
CSV (or stdout) with one row per residue entry.
"""
from __future__ import annotations

import argparse
import csv
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator, Optional


@dataclass
class PkaRecord:
    """Structured representation of a PROPKA summary row."""

    source_file: Path
    residue_name: str
    residue_number: str
    chain_id: str
    predicted_pka: Optional[float]
    model_pka: Optional[float]
    ligand: Optional[str]
    atom_type: Optional[str]


SUMMARY_HEADER = "SUMMARY OF THIS PREDICTION"
TABLE_HEADER_KEYWORD = "Group"


def iter_pka_files(root: Path, recursive: bool = True) -> Iterator[Path]:
    """Yield `.pka` files under *root*."""

    pattern = "**/*.pka" if recursive else "*.pka"
    yield from root.glob(pattern)


def parse_summary_block(lines: Iterable[str]) -> Iterator[str]:
    """Yield lines belonging to the summary table."""

    lines = list(lines)
    try:
        summary_idx = next(
            idx for idx, line in enumerate(lines) if SUMMARY_HEADER in line
        )
    except StopIteration as exc:
        raise ValueError("Summary header not found") from exc

    # Find the table header following the summary marker.
    try:
        header_idx = next(
            idx
            for idx in range(summary_idx + 1, len(lines))
            if TABLE_HEADER_KEYWORD in lines[idx]
        )
    except StopIteration as exc:
        raise ValueError("Table header following summary not found") from exc

    # Data begins on the next line after the header.
    for line in lines[header_idx + 1 :]:
        stripped = line.strip()
        if not stripped:
            break
        if set(stripped) == {"-"}:
            break
        yield line


def parse_record(source: Path, line: str) -> PkaRecord:
    """Convert a summary line into a `PkaRecord`."""

    def to_float(value: str) -> Optional[float]:
        if not value or value in {"NA", "N/A", "nan"}:
            return None
        try:
            return float(value)
        except ValueError:
            return None

    # Split the line by whitespace
    tokens = line.split()
    
    # Expected format: RESIDUE_NAME RESIDUE_NUMBER CHAIN pKa model-pKa [ligand] [atom-type]
    residue_name = tokens[0] if len(tokens) > 0 else ""
    residue_number = tokens[1] if len(tokens) > 1 else ""
    chain_id = tokens[2] if len(tokens) > 2 else ""
    pka_str = tokens[3] if len(tokens) > 3 else ""
    model_str = tokens[4] if len(tokens) > 4 else ""
    ligand = tokens[5] if len(tokens) > 5 else None
    atom_type = tokens[6] if len(tokens) > 6 else None

    predicted = to_float(pka_str)
    model = to_float(model_str)

    return PkaRecord(
        source_file=source,
        residue_name=residue_name,
        residue_number=residue_number,
        chain_id=chain_id,
        predicted_pka=predicted,
        model_pka=model,
        ligand=ligand,
        atom_type=atom_type,
    )


def extract_records(path: Path) -> list[PkaRecord]:
    """Extract summary entries from a single `.pka` file."""

    text = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        lines = list(parse_summary_block(text))
    except ValueError:
        return []
    return [parse_record(path, line) for line in lines]


def write_records(records: Iterable[PkaRecord], output: Optional[Path]) -> None:
    """Write records to *output* (CSV file or stdout)."""

    fieldnames = [
        "source_file",
        "residue_name",
        "residue_number",
        "chain_id",
        "predicted_pka",
        "model_pka",
        "ligand",
        "atom_type",
    ]

    writer: csv.DictWriter
    if output:
        output.parent.mkdir(parents=True, exist_ok=True)
        handle = output.open("w", newline="", encoding="utf-8")
        close_handle = True
    else:
        handle = sys.stdout
        close_handle = False

    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for record in records:
        writer.writerow(
            {
                "source_file": str(record.source_file),
                "residue_name": record.residue_name,
                "residue_number": record.residue_number,
                "chain_id": record.chain_id,
                "predicted_pka": record.predicted_pka,
                "model_pka": record.model_pka,
                "ligand": record.ligand,
                "atom_type": record.atom_type,
            }
        )

    if close_handle:
        handle.close()


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Extract predicted pKa values from PROPKA .pka files.",
    )
    parser.add_argument(
        "input_dir",
        type=Path,
        help="Directory containing PROPKA .pka outputs.",
    )
    parser.add_argument(
        "--no-recursive",
        dest="recursive",
        action="store_false",
        help="Only scan the top-level directory (no subdirectories).",
    )
    parser.add_argument(
        "-o",
        "--output",
        type=Path,
        help="Optional path to write the consolidated CSV (defaults to stdout).",
    )
    return parser


def main(argv: Optional[list[str]] = None) -> int:
    # Hardcoded paths
    input_dir = Path("/home/ziyu-song/Graph_pKa/Mutant_Data/PROPKA_Results/")
    output_file = Path("/home/ziyu-song/Graph_pKa/Mutant_Data/pka_results.csv")

    if not input_dir.exists():
        print(f"Error: Input directory '{input_dir}' does not exist.")
        return 1
    if not input_dir.is_dir():
        print(f"Error: Input path '{input_dir}' is not a directory.")
        return 1

    files = sorted(iter_pka_files(input_dir, recursive=True))
    if not files:
        print("Error: No .pka files found in the provided directory.")
        return 1

    all_records: list[PkaRecord] = []
    for path in files:
        all_records.extend(extract_records(path))

    if not all_records:
        print("Error: No summary records extracted from the discovered .pka files.")
        return 1

    write_records(all_records, output_file)
    print(f"Successfully extracted {len(all_records)} records to {output_file}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [8]:
import pandas as pd
from pathlib import Path

# Extract DeepKa predictions from CSV files
deepka_dir = Path("/home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Prediction/")

# Read all CSV files from the DeepKa directory
deepka_files = list(deepka_dir.glob("*.csv"))
print(f"Found {len(deepka_files)} CSV files in {deepka_dir}")

# Combine all CSV files into a single dataframe
deepka_data = []
for csv_file in sorted(deepka_files):
    # Try comma separator first, then tab
    df = pd.read_csv(csv_file, sep=',')
    df['source_file'] = csv_file.name
    deepka_data.append(df)

# Combine all dataframes
deepka_combined = pd.concat(deepka_data, ignore_index=True)

# First, let's check what columns we actually have
print(f"\nActual columns in dataframe: {list(deepka_combined.columns)}")

# Rename columns to match PROPKA format
deepka_combined = deepka_combined.rename(columns={
    'PDB ID': 'pdb_id',
    'Res ID': 'residue_number',
    'Res Name': 'residue_name',
    'Chain': 'chain_id',
    'model pKa': 'model_pka',
    'Predict pKa': 'predicted_pka'
})

print(f"Columns after renaming: {list(deepka_combined.columns)}")

# Keep only the required columns that exist
available_cols = [col for col in ['source_file', 'pdb_id', 'residue_name', 'residue_number', 'chain_id', 'predicted_pka', 'model_pka'] if col in deepka_combined.columns]
deepka_combined = deepka_combined[available_cols]

# Save to CSV
deepka_combined.to_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/deepka_results.csv', index=False)

print(f"\nLoaded {len(deepka_combined)} records from DeepKa predictions")
print(f"Columns: {list(deepka_combined.columns)}")
print(f"\nFirst few rows:")
print(deepka_combined.head(10))

Found 45 CSV files in /home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Prediction

Actual columns in dataframe: ['Model', 'PDB ID', 'Chain', 'Res ID', 'Res Name', 'model pKa', 'Predict pKa', 'Predict pKa shift', 'source_file']
Columns after renaming: ['Model', 'pdb_id', 'chain_id', 'residue_number', 'residue_name', 'model_pka', 'predicted_pka', 'Predict pKa shift', 'source_file']

Loaded 1719 records from DeepKa predictions
Columns: ['source_file', 'pdb_id', 'residue_name', 'residue_number', 'chain_id', 'predicted_pka', 'model_pka']

First few rows:
  source_file pdb_id residue_name  residue_number chain_id  predicted_pka  \
0    1BVC.csv   1bvc          GLU               4        A           3.42   
1    1BVC.csv   1bvc          GLU               6        A           3.41   
2    1BVC.csv   1bvc          HIS              12        A           6.43   
3    1BVC.csv   1bvc          LYS              16        A          10.44   
4    1BVC.csv   1bvc          GLU              18        A     

In [18]:
# Merge PROPKA and DeepKa results
propka_df = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/pka_results.csv')
deepka_df = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/deepka_results.csv')

print(f"PROPKA records: {len(propka_df)}")
print(f"DeepKa records: {len(deepka_df)}")

# Extract PDB ID from source_file path for PROPKA
# Format: /home/ziyu-song/Graph_pKa/Mutant_Data/PROPKA_Results/1BVC.pka
propka_df['pdb_id'] = propka_df['source_file'].apply(lambda x: Path(x).stem.upper())
deepka_df['pdb_id'] = deepka_df['pdb_id'].astype(str).str.upper()

print(f"\nPROPKA columns: {list(propka_df.columns)}")
print(f"Sample PROPKA pdb_ids: {propka_df['pdb_id'].head()}")

# Rename predicted_pka to avoid column name conflict after merge
propka_df = propka_df.rename(columns={
    'predicted_pka': 'propka_predicted_pka'
})

deepka_df = deepka_df.rename(columns={
    'predicted_pka': 'deepka_predicted_pka'
})

# Convert residue_number to string for both dataframes to ensure matching types
propka_df['residue_number'] = propka_df['residue_number'].astype(str)
deepka_df['residue_number'] = deepka_df['residue_number'].astype(str)

# Merge on pdb_id, residue_name, residue_number, chain_id
merge_keys = ['pdb_id', 'residue_name', 'residue_number', 'chain_id']

print(f"\nDeepKa columns: {list(deepka_df.columns)}")
print(f"PROPKA columns (after renaming): {list(propka_df.columns)}")

# Check common PDB IDs now
common_pdbs = set(propka_df['pdb_id'].unique()) & set(deepka_df['pdb_id'].unique())
print(f"Common PDB IDs after case normalization: {sorted(common_pdbs)[:10]}")

# Merge the dataframes
merged = pd.merge(
    propka_df,
    deepka_df,
    on=merge_keys,
    how='inner',  # Inner join to keep only matching records
    suffixes=('_propka', '_deepka')
)

# Filter to keep only rows that have BOTH predicted pKa values
merged = merged[
    (merged['propka_predicted_pka'].notna()) & 
    (merged['deepka_predicted_pka'].notna())
]

print(f"\nMerged records (with both predictions): {len(merged)}")

# Select and reorder columns
output_columns = [
    'pdb_id', 'residue_name', 'residue_number', 'chain_id',
     'propka_predicted_pka',
     'deepka_predicted_pka'
]

# Only keep columns that exist
output_columns = [col for col in output_columns if col in merged.columns]
merged = merged[output_columns]

# Save the merged file
merged.to_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Propka_predictions.csv', index=False)

print(f"\nMerged file saved to: /home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Propka_predictions.csv")
print(f"Columns: {list(merged.columns)}")
print(f"\nFirst few rows:")
print(merged.head(10))

PROPKA records: 3218
DeepKa records: 1719

PROPKA columns: ['source_file', 'residue_name', 'residue_number', 'chain_id', 'predicted_pka', 'model_pka', 'ligand', 'atom_type', 'pdb_id']
Sample PROPKA pdb_ids: 0    1BVC
1    1BVC
2    1BVC
3    1BVC
4    1BVC
Name: pdb_id, dtype: object

DeepKa columns: ['source_file', 'pdb_id', 'residue_name', 'residue_number', 'chain_id', 'deepka_predicted_pka', 'model_pka']
PROPKA columns (after renaming): ['source_file', 'residue_name', 'residue_number', 'chain_id', 'propka_predicted_pka', 'model_pka', 'ligand', 'atom_type', 'pdb_id']
Common PDB IDs after case normalization: ['1BVC', '1EY7', '1L98', '1LE2', '1PRW', '1QT8', '1VII', '2OEO', '2OXP', '2QDB']

Merged records (with both predictions): 1707

Merged file saved to: /home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Propka_predictions.csv
Columns: ['pdb_id', 'residue_name', 'residue_number', 'chain_id', 'propka_predicted_pka', 'deepka_predicted_pka']

First few rows:
  pdb_id residue_name residue_numb

In [22]:
# Load and merge with the merged_predictions_dataset CSVs (only dataset 3)
import glob

# Find only the dataset 3 merged_predictions file
predictions_dir = Path("/home/ziyu-song/Graph_pKa/Mutant_Data/All_Predictions/Merged/")
prediction_file = predictions_dir / "merged_predictions_dataset_3.csv"

print(f"Loading GAT predictions from: {prediction_file.name}")

# Load dataset 3
if prediction_file.exists():
    all_predictions = pd.read_csv(prediction_file)
else:
    print(f"Error: {prediction_file} not found!")
    all_predictions = pd.DataFrame()

print(f"Loaded {len(all_predictions)} records from GAT dataset 3")
print(f"Columns: {list(all_predictions.columns)}")
print(f"\nFirst few rows:")
print(all_predictions.head())

# Rename Average_Predicted_pKa to GAT_Ave_Prediction
all_predictions = all_predictions.rename(columns={
    'Average_Predicted_pKa': 'GAT_Ave_Prediction'
})

# Read the current merged data
current_merged = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/DeepKa_Propka_predictions.csv')

print(f"\n\nCurrent merged predictions: {len(current_merged)} records")
print(f"Columns: {list(current_merged.columns)}")

# Prepare columns for merge
# Rename to match the merge keys
current_merged = current_merged.rename(columns={
    'pdb_id': 'PDB_ID',
    'residue_number': 'Residue_Number',
    'residue_name': 'Residue'
})

# Merge on PDB_ID, Residue_Number, Residue
merge_keys = ['PDB_ID', 'Residue_Number', 'Residue']

# Check if all keys exist in both dataframes
current_cols = set(current_merged.columns)
predictions_cols = set(all_predictions.columns)

print(f"\nMerge keys needed: {merge_keys}")
print(f"Available in current_merged: {[k for k in merge_keys if k in current_cols]}")
print(f"Available in all_predictions: {[k for k in merge_keys if k in predictions_cols]}")

# Perform the merge
final_merged = pd.merge(
    current_merged,
    all_predictions,
    on=merge_keys,
    how='left',  # Left join to keep all current predictions
    suffixes=('_propka_deepka', '_gat')
)

print(f"\nFinal merged records (before filtering): {len(final_merged)}")

# Filter to keep only rows with ALL required columns having values
required_columns = ['propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction']

# Check which required columns exist
existing_required_cols = [col for col in required_columns if col in final_merged.columns]
print(f"Required columns found: {existing_required_cols}")

# Filter rows where all required columns have values (not NaN)
final_merged_filtered = final_merged.dropna(subset=existing_required_cols)

print(f"Final merged records (after filtering for all values present): {len(final_merged_filtered)}")

# Select and reorder columns - keep PDB_ID, Residue info plus required columns
output_columns = ['PDB_ID', 'Residue', 'Residue_Number', 'chain_id'] + existing_required_cols
output_columns = [col for col in output_columns if col in final_merged_filtered.columns]

final_merged_filtered = final_merged_filtered[output_columns]

# Save the final merged file
final_merged_filtered.to_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_with_GAT.csv', index=False)

print(f"\nFinal merged file saved to: /home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_with_GAT.csv")
print(f"Columns: {list(final_merged_filtered.columns)}")
print(f"\nFirst few rows:")
print(final_merged_filtered.head(10))

Loading GAT predictions from: merged_predictions_dataset_3.csv
Loaded 157 records from GAT dataset 3
Columns: ['PDB_ID', 'Residue_Number', 'Residue', 'True_pKa', 'Average_Predicted_pKa']

First few rows:
  PDB_ID  Residue_Number Residue  True_pKa  Average_Predicted_pKa
0   2RDF             121     HIS      4.24               5.950692
1   3BDC             101     GLU      3.81               3.317887
2   1PRW              94     LYS      9.65              10.486496
3   1BVC             113     HIS      5.50               6.080379
4   2RVQ              92     GLU      4.40               4.851661


Current merged predictions: 1707 records
Columns: ['pdb_id', 'residue_name', 'residue_number', 'chain_id', 'propka_predicted_pka', 'deepka_predicted_pka']

Merge keys needed: ['PDB_ID', 'Residue_Number', 'Residue']
Available in current_merged: ['PDB_ID', 'Residue_Number', 'Residue']
Available in all_predictions: ['PDB_ID', 'Residue_Number', 'Residue']

Final merged records (before filtering): 17

In [24]:
# Filter Final_Predictions_with_GAT.csv to keep only rows that ARE in Node_Feature_Vectors_Summary_7.csv

# Load the final predictions
final_predictions = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_with_GAT.csv')
print(f"Final predictions records: {len(final_predictions)}")
print(f"Columns: {list(final_predictions.columns)}")

# Load the node feature vectors summary
node_features_path = Path("/home/ziyu-song/Graph_pKa/Mutant_Data/Node_Feature_Vectors_Summary_7.csv")
if node_features_path.exists():
    node_features = pd.read_csv(node_features_path)
    print(f"\nNode feature vectors records: {len(node_features)}")
    print(f"Columns: {list(node_features.columns)}")
else:
    print(f"Error: {node_features_path} not found!")
    node_features = pd.DataFrame()

# Normalize column names for matching - check what columns are available
print(f"\nFinal predictions sample:")
print(final_predictions.head(2))
print(f"\nNode features sample:")
print(node_features.head(2))

# Create matching keys - need to identify the matching column names
# Node features has: PDB, Residue Name, Residue_Number
# Final predictions has: PDB_ID, Residue, Residue_Number

# Create a merge key dataframe from node_features to identify which rows to include
if not node_features.empty:
    # Normalize node features columns to match final predictions
    node_features_normalized = node_features.copy()
    
    # Create a compound key for matching
    node_features_normalized['match_key'] = (
        node_features_normalized['PDB'].astype(str).str.upper() + '_' +
        node_features_normalized['Residue Name'].astype(str) + '_' +
        node_features_normalized['Residue_Number'].astype(str)
    )
    
    # Create matching key in final predictions
    final_predictions['match_key'] = (
        final_predictions['PDB_ID'].astype(str).str.upper() + '_' +
        final_predictions['Residue'].astype(str) + '_' +
        final_predictions['Residue_Number'].astype(str)
    )
    
    # Find rows in final_predictions that ARE in node_features
    included_keys = set(node_features_normalized['match_key'].unique())
    predictions_in_features = final_predictions[final_predictions['match_key'].isin(included_keys)].copy()
    
    # Remove the temporary match_key column
    predictions_in_features = predictions_in_features.drop(columns=['match_key'])
    
    print(f"\nFinal predictions records in Node Features: {len(predictions_in_features)}")
    print(f"Final predictions records NOT in Node Features: {len(final_predictions) - len(predictions_in_features)}")
    
    # Save the filtered results
    output_path = '/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_in_Node_Features.csv'
    predictions_in_features.to_csv(output_path, index=False)
    
    print(f"\nFiltered predictions saved to: {output_path}")
    print(f"Columns: {list(predictions_in_features.columns)}")
    print(f"\nFirst few rows:")
    print(predictions_in_features.head(10))
else:
    print("Node features dataframe is empty. Skipping filter.")

Final predictions records: 135
Columns: ['PDB_ID', 'Residue', 'Residue_Number', 'chain_id', 'propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction']

Node feature vectors records: 97
Columns: ['PDB', 'Residue Name', 'Chain ID', 'Residue_Number']

Final predictions sample:
  PDB_ID Residue  Residue_Number chain_id  propka_predicted_pka  \
0   1BVC     HIS              24        A                  4.31   
1   1BVC     HIS              36        A                  6.15   

   deepka_predicted_pka  True_pKa  GAT_Ave_Prediction  
0                  3.49       4.8            5.726973  
1                  7.99       8.2            8.147904  

Node features sample:
    PDB Residue Name Chain ID  Residue_Number
0  1BVC          HIS  Unknown              24
1  1BVC          HIS  Unknown              36

Final predictions records in Node Features: 83
Final predictions records NOT in Node Features: 52

Filtered predictions saved to: /home/ziyu-song/Graph_pKa/Mutant_Data/Fi

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Load the final predictions with all models
final_predictions = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_in_Node_Features.csv')

print(f"Total records for evaluation: {len(final_predictions)}")
print(f"Columns: {list(final_predictions.columns)}\n")

# Get the true pKa values
true_pka = final_predictions['True_pKa'].values

# Calculate metrics for each model
metrics = []

# PROPKA predictions
if 'propka_predicted_pka' in final_predictions.columns:
    propka_pred = final_predictions['propka_predicted_pka'].values
    propka_mae = mean_absolute_error(true_pka, propka_pred)
    propka_rmse = np.sqrt(mean_squared_error(true_pka, propka_pred))
    metrics.append({
        'Model': 'PROPKA',
        'MAE': propka_mae,
        'RMSE': propka_rmse,
        'Samples': len(final_predictions)
    })
    print(f"PROPKA:")
    print(f"  MAE:  {propka_mae:.4f}")
    print(f"  RMSE: {propka_rmse:.4f}\n")

# DeepKa predictions
if 'deepka_predicted_pka' in final_predictions.columns:
    deepka_pred = final_predictions['deepka_predicted_pka'].values
    deepka_mae = mean_absolute_error(true_pka, deepka_pred)
    deepka_rmse = np.sqrt(mean_squared_error(true_pka, deepka_pred))
    metrics.append({
        'Model': 'DeepKa',
        'MAE': deepka_mae,
        'RMSE': deepka_rmse,
        'Samples': len(final_predictions)
    })
    print(f"DeepKa:")
    print(f"  MAE:  {deepka_mae:.4f}")
    print(f"  RMSE: {deepka_rmse:.4f}\n")

# GAT predictions
if 'GAT_Ave_Prediction' in final_predictions.columns:
    gat_pred = final_predictions['GAT_Ave_Prediction'].values
    gat_mae = mean_absolute_error(true_pka, gat_pred)
    gat_rmse = np.sqrt(mean_squared_error(true_pka, gat_pred))
    metrics.append({
        'Model': 'GAT',
        'MAE': gat_mae,
        'RMSE': gat_rmse,
        'Samples': len(final_predictions)
    })
    print(f"GAT:")
    print(f"  MAE:  {gat_mae:.4f}")
    print(f"  RMSE: {gat_rmse:.4f}\n")

# Create a summary dataframe
metrics_df = pd.DataFrame(metrics)

print("=" * 50)
print("Summary of Model Performance")
print("=" * 50)
print(metrics_df.to_string(index=False))

# Save metrics to CSV
metrics_output_path = '/home/ziyu-song/Graph_pKa/Mutant_Data/Model_Performance_Metrics.csv'
metrics_df.to_csv(metrics_output_path, index=False)
print(f"\nMetrics saved to: {metrics_output_path}")


Total records for evaluation: 83
Columns: ['PDB_ID', 'Residue', 'Residue_Number', 'chain_id', 'propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction']

PROPKA:
  MAE:  0.7352
  RMSE: 1.0923

DeepKa:
  MAE:  0.7594
  RMSE: 1.0379

GAT:
  MAE:  0.7207
  RMSE: 0.9825

Summary of Model Performance
 Model      MAE     RMSE  Samples
PROPKA 0.735181 1.092307       83
DeepKa 0.759398 1.037854       83
   GAT 0.720701 0.982452       83

Metrics saved to: /home/ziyu-song/Graph_pKa/Mutant_Data/Model_Performance_Metrics.csv


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Load the final predictions with all models
final_predictions = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_in_Node_Features.csv')

print(f"Total records for evaluation: {len(final_predictions)}")
print(f"Columns: {list(final_predictions.columns)}\n")

# Get the true pKa values
true_pka = final_predictions['True_pKa'].values

# Function to calculate metrics safely (handles NaN values)
def calc_metrics(y_true, y_pred, model_name):
    """Calculate MAE, RMSE with NaN handling"""
    # Remove NaN values
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]
    
    if len(y_true_clean) == 0:
        return None
    
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    
    return {
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'Samples': len(y_true_clean),
        'Mean_Pred': np.mean(y_pred_clean),
        'Std_Pred': np.std(y_pred_clean)
    }

# Calculate overall metrics for each model
metrics = []

# PROPKA predictions
if 'propka_predicted_pka' in final_predictions.columns:
    result = calc_metrics(true_pka, final_predictions['propka_predicted_pka'].values, 'PROPKA')
    if result:
        metrics.append(result)
        print(f"PROPKA:")
        print(f"  Samples: {result['Samples']}")
        print(f"  MAE:  {result['MAE']:.4f}")
        print(f"  RMSE: {result['RMSE']:.4f}\n")

# DeepKa predictions
if 'deepka_predicted_pka' in final_predictions.columns:
    result = calc_metrics(true_pka, final_predictions['deepka_predicted_pka'].values, 'DeepKa')
    if result:
        metrics.append(result)
        print(f"DeepKa:")
        print(f"  Samples: {result['Samples']}")
        print(f"  MAE:  {result['MAE']:.4f}")
        print(f"  RMSE: {result['RMSE']:.4f}\n")

# GAT predictions
if 'GAT_Ave_Prediction' in final_predictions.columns:
    result = calc_metrics(true_pka, final_predictions['GAT_Ave_Prediction'].values, 'GAT')
    if result:
        metrics.append(result)
        print(f"GAT:")
        print(f"  Samples: {result['Samples']}")
        print(f"  MAE:  {result['MAE']:.4f}")
        print(f"  RMSE: {result['RMSE']:.4f}\n")

# Create summary dataframe
metrics_df = pd.DataFrame(metrics)

print("=" * 60)
print("Summary of Overall Model Performance")
print("=" * 60)
print(metrics_df.to_string(index=False))

# Per-residue analysis
print("\n" + "=" * 60)
print("Per-Residue Performance Analysis")
print("=" * 60)

residue_metrics = []
for residue in sorted(final_predictions['Residue'].unique()):
    residue_data = final_predictions[final_predictions['Residue'] == residue]
    residue_true = residue_data['True_pKa'].values
    
    print(f"\n{residue} (n={len(residue_data)}):")
    
    if 'propka_predicted_pka' in residue_data.columns:
        result = calc_metrics(residue_true, residue_data['propka_predicted_pka'].values, f'{residue}_PROPKA')
        if result:
            residue_metrics.append(result)
            print(f"  PROPKA: MAE={result['MAE']:.4f}, RMSE={result['RMSE']:.4f}")
    
    if 'deepka_predicted_pka' in residue_data.columns:
        result = calc_metrics(residue_true, residue_data['deepka_predicted_pka'].values, f'{residue}_DeepKa')
        if result:
            residue_metrics.append(result)
            print(f"  DeepKa: MAE={result['MAE']:.4f}, RMSE={result['RMSE']:.4f}")
    
    if 'GAT_Ave_Prediction' in residue_data.columns:
        result = calc_metrics(residue_true, residue_data['GAT_Ave_Prediction'].values, f'{residue}_GAT')
        if result:
            residue_metrics.append(result)
            print(f"  GAT:    MAE={result['MAE']:.4f}, RMSE={result['RMSE']:.4f}")

# Save metrics to CSV
residue_metrics_df = pd.DataFrame(residue_metrics)
metrics_output_path = '/home/ziyu-song/KaMLs/Mutant_For_tree/Model_Performance_Metrics.csv'
residue_metrics_df.to_csv(metrics_output_path, index=False)

print("\n" + "=" * 60)
print(f"Detailed metrics saved to: {metrics_output_path}")
print("=" * 60)

# Display top performers
print("\nBest Performing Models (Overall):")
best_mae_idx = metrics_df['MAE'].idxmin()
best_rmse_idx = metrics_df['RMSE'].idxmin()
print(f"  Best MAE:  {metrics_df.loc[best_mae_idx, 'Model']} ({metrics_df.loc[best_mae_idx, 'MAE']:.4f})")
print(f"  Best RMSE: {metrics_df.loc[best_rmse_idx, 'Model']} ({metrics_df.loc[best_rmse_idx, 'RMSE']:.4f})")

Total records for evaluation: 83
Columns: ['PDB_ID', 'Residue', 'Residue_Number', 'chain_id', 'propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction']

PROPKA:
  Samples: 83
  MAE:  0.7352
  RMSE: 1.0923

DeepKa:
  Samples: 83
  MAE:  0.7594
  RMSE: 1.0379

GAT:
  Samples: 83
  MAE:  0.7207
  RMSE: 0.9825

Summary of Overall Model Performance
 Model      MAE     RMSE  Samples  Mean_Pred  Std_Pred
PROPKA 0.735181 1.092307       83   5.619398  2.532583
DeepKa 0.759398 1.037854       83   5.676386  2.664788
   GAT 0.720701 0.982452       83   5.768614  2.514007

Per-Residue Performance Analysis

ASP (n=15):
  PROPKA: MAE=1.3187, RMSE=1.9585
  DeepKa: MAE=1.2660, RMSE=1.7394
  GAT:    MAE=1.0954, RMSE=1.5399

GLU (n=32):
  PROPKA: MAE=0.6472, RMSE=0.7964
  DeepKa: MAE=0.4787, RMSE=0.5577
  GAT:    MAE=0.4465, RMSE=0.5807

HIS (n=21):
  PROPKA: MAE=0.6848, RMSE=0.8736
  DeepKa: MAE=0.7729, RMSE=0.9493
  GAT:    MAE=0.8405, RMSE=1.0076

LYS (n=15):
  PROPKA: MAE=0.41

In [5]:
# Merge final predictions with averaged dataset predictions
import pandas as pd

print("\n" + "=" * 80)
print("Merging Final Predictions with Dataset Averaged Predictions")
print("=" * 80)

# Load the two files
final_predictions = pd.read_csv('/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_in_Node_Features.csv')
averaged_predictions = pd.read_csv('/home/ziyu-song/Graph_pKa/Results/Predictions/Dataset_Averaged_Predictions/predictions_dataset_3_averaged.csv')

print(f"\nFinal Predictions shape: {final_predictions.shape}")
print(f"Columns: {list(final_predictions.columns)}")
print(f"Sample Residue values: {final_predictions['Residue'].unique()[:5]}")

print(f"\nAveraged Predictions shape: {averaged_predictions.shape}")
print(f"Columns: {list(averaged_predictions.columns)}")
print(f"Sample Residue values: {averaged_predictions['Residue'].unique()[:5]}")

# Create mapping from 3-letter to full name (matching run_Tinker_Output_Processing.py)
residue_3_to_full = {
    "ASP": "Aspartate",
    "GLU": "Glutamate",
    "LYS": "Lysine",
    "HIS": "Histidine",
}

# Convert final_predictions residues from 3-letter to full name
final_predictions_copy = final_predictions.copy()
final_predictions_copy['Residue'] = final_predictions_copy['Residue'].map(residue_3_to_full)

print(f"\nAfter conversion - Final Predictions Residue values: {final_predictions_copy['Residue'].unique()}")

# Merge on common identifiers: PDB_ID, Chain_ID, Residue_Number, Residue (now matching)
if 'PDB_ID' in final_predictions_copy.columns and 'PDB_ID' in averaged_predictions.columns:
    merged_df = pd.merge(
        final_predictions_copy,
        averaged_predictions,
        on=['PDB_ID', 'Chain_ID', 'Residue_Number', 'Residue'],
        how='inner',
        suffixes=('_final', '_averaged')
    )
    
    print(f"\nMerged DataFrame shape: {merged_df.shape}")
    print(f"Merged Columns: {list(merged_df.columns)}")
    print(f"\nFirst few rows of merged data:")
    print(merged_df.head())
    
    # Save merged results
    output_path = '/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_Merged_with_Averaged.csv'
    merged_df.to_csv(output_path, index=False)
    print(f"\nMerged predictions saved to: {output_path}")
    print(f"Successfully merged {len(merged_df)} records")
    
    # Save filtered final_predictions (only entries that matched)
    matched_keys = set(merged_df[['PDB_ID', 'Chain_ID', 'Residue_Number', 'Residue']].apply(tuple, axis=1))
    final_predictions_matched = final_predictions_copy[
        final_predictions_copy[['PDB_ID', 'Chain_ID', 'Residue_Number', 'Residue']].apply(tuple, axis=1).isin(matched_keys)
    ].copy()
    
    # Save filtered final predictions
    filtered_output_path = '/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_Filtered_Matched.csv'
    final_predictions_matched.to_csv(filtered_output_path, index=False)
    print(f"\nFiltered Final Predictions (matched entries) saved to: {filtered_output_path}")
    print(f"Total matched entries: {len(final_predictions_matched)}")
    
    # Also save unmatched entries for reference
    final_predictions_unmatched = final_predictions_copy[
        ~final_predictions_copy[['PDB_ID', 'Chain_ID', 'Residue_Number', 'Residue']].apply(tuple, axis=1).isin(matched_keys)
    ].copy()
    
    unmatched_output_path = '/home/ziyu-song/Graph_pKa/Mutant_Data/Final_Predictions_Filtered_Unmatched.csv'
    final_predictions_unmatched.to_csv(unmatched_output_path, index=False)
    print(f"\nUnmatched Final Predictions saved to: {unmatched_output_path}")
    print(f"Total unmatched entries: {len(final_predictions_unmatched)}")
    
    print(f"\n" + "=" * 80)
    print(f"Summary:")
    print(f"  Total final predictions: {len(final_predictions_copy)}")
    print(f"  Matched with averaged: {len(final_predictions_matched)}")
    print(f"  Unmatched: {len(final_predictions_unmatched)}")
    print(f"=" * 80)
    
else:
    print("\nError: Could not find required columns for merging")
    print(f"Final Predictions columns: {list(final_predictions_copy.columns)}")
    print(f"Averaged Predictions columns: {list(averaged_predictions.columns)}")


Merging Final Predictions with Dataset Averaged Predictions

Final Predictions shape: (83, 8)
Columns: ['PDB_ID', 'Residue', 'Residue_Number', 'Chain_ID', 'propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction']
Sample Residue values: ['HIS' 'GLU' 'LYS' 'ASP']

Averaged Predictions shape: (2274, 5)
Columns: ['PDB_ID', 'Chain_ID', 'Residue_Number', 'Residue', 'Predicted_pKa']
Sample Residue values: ['Glutamate' 'Histidine' 'Lysine' 'Aspartate']

After conversion - Final Predictions Residue values: ['Histidine' 'Glutamate' 'Lysine' 'Aspartate']

Merged DataFrame shape: (38, 9)
Merged Columns: ['PDB_ID', 'Residue', 'Residue_Number', 'Chain_ID', 'propka_predicted_pka', 'deepka_predicted_pka', 'True_pKa', 'GAT_Ave_Prediction', 'Predicted_pKa']

First few rows of merged data:
  PDB_ID    Residue  Residue_Number Chain_ID  propka_predicted_pka  \
0   1BVC  Histidine              24        A                  4.31   
1   1BVC  Histidine              36        A         